# What-If Tool Demonstration
This notebook demonstrates how to use the Google What-If Tool for analyzing model fairness.

**Important**: This notebook must be run in **Jupyter Notebook Classic** (nbclassic) interface — not JupyterLab or VS Code.
The WIT widget relies on nbextension-based rendering which is not compatible with JupyterLab 4.x.

In [1]:
#!pip install witwidget

In [2]:
#!pip install -r ../requirements.txt
#!pip install pandas
#!pip install scikit-learn
#!pip install witwidget

In [3]:
import os
import warnings

# CRITICAL: These env vars MUST be set before importing TF or witwidget
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
warnings.filterwarnings('ignore', category=FutureWarning)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

print('Core libraries loaded successfully.')

Core libraries loaded successfully.


In [4]:
# Import WIT widget
from witwidget.notebook.visualization import WitWidget, WitConfigBuilder
print('WitWidget imported successfully.')

WitWidget imported successfully.


## 1. Generate Synthetic Data
We create a loan approval dataset with a protected attribute (Demographic Group) to simulate historical bias.

In [5]:
def create_synthetic_data(n_samples=1000):
    """
    Generates a synthetic dataset for loan approvals with a protected attribute.
    This simulates a scenario where bias might exist, ideal for stress-testing.
    """
    np.random.seed(42)
    
    # Basic features
    age = np.random.randint(18, 70, n_samples)
    income = np.random.normal(60000, 20000, n_samples)
    credit_score = np.random.normal(650, 80, n_samples)
    
    # Protected attribute (Demographic: Group A vs Group B)
    # Simulating a historical disadvantage for Group B
    group = np.random.choice(['Group A', 'Group B'], size=n_samples, p=[0.7, 0.3])
    
    income = np.where(group == 'Group B', income - 5000, income)
    credit_score = np.where(group == 'Group B', credit_score - 20, credit_score)
    
    # Target variable: Loan Approved (1) or Denied (0)
    prob = (income / 100000) * 0.4 + (credit_score / 800) * 0.6
    prob += np.random.normal(0, 0.1, n_samples)
    
    approved = (prob > 0.65).astype(int)
    
    df = pd.DataFrame({
        'Age': age,
        'Income': income,
        'Credit_Score': credit_score,
        'Demographic_Group': group,
        'Loan_Approved': approved
    })
    
    return df

df = create_synthetic_data()
print(f'Dataset created: {df.shape[0]} samples, {df.shape[1]} features')
df.head()

Dataset created: 1000 samples, 5 features


,Age,Income,Credit_Score,Demographic_Group,Loan_Approved
0,56,27871.073595,788.157108,Group A,1
1,69,64069.272717,618.029105,Group A,1
2,46,39872.985094,647.974778,Group B,1
3,32,31554.925808,724.607267,Group A,1
4,60,42068.542315,516.530741,Group B,0


## 2. Train Model

In [6]:
def train_model(df):
    """Trains a Random Forest classifier to predict loan approvals."""
    df_encoded = df.copy()
    df_encoded['Demographic_Group'] = df_encoded['Demographic_Group'].map({'Group A': 0, 'Group B': 1})
    
    X = df_encoded.drop('Loan_Approved', axis=1)
    y = df_encoded['Loan_Approved']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    accuracy = accuracy_score(y_test, model.predict(X_test))
    print(f'Model Accuracy: {accuracy:.3f}')
    
    return model, X_test, y_test, df.iloc[X_test.index]

model, X_test, y_test, df_test_original = train_model(df)

Model Accuracy: 0.730


## 3. Configure and Launch the What-If Tool

The What-If Tool provides an interactive visual interface to:
- Slice data by `Demographic_Group` to compare outcomes
- Adjust decision thresholds and observe impact on fairness
- Explore fairness metrics like Equal Opportunity
- Edit individual data points and observe prediction changes

In [7]:
# Prepare examples as list of dicts (WIT format)
examples = df_test_original.to_dict(orient='records')

# Custom prediction function that WIT calls interactively
def custom_predict(examples_to_predict):
    df_pred = pd.DataFrame(examples_to_predict)
    df_pred['Demographic_Group'] = df_pred['Demographic_Group'].map({'Group A': 0, 'Group B': 1})
    features = ['Age', 'Income', 'Credit_Score', 'Demographic_Group']
    X_pred = df_pred[features]
    preds = model.predict_proba(X_pred)
    return preds.tolist()

# Configure WIT (limit to 200 examples for responsiveness)
config_builder = (WitConfigBuilder(examples[:200])
                  .set_custom_predict_fn(custom_predict)
                  .set_target_feature('Loan_Approved')
                  .set_label_vocab(['Denied', 'Approved']))

print('WIT configuration ready. Rendering widget below...')

WIT configuration ready. Rendering widget below...


In [8]:
# Render the WIT widget — this MUST be the last expression in the cell
widget = WitWidget(config_builder, height=800)
widget

WitWidget(config={'model_type': 'classification', 'label_vocab': ['Denied', 'Approved'], 'uses_json_input': Tr…